Reading silver data

In [0]:
silver_df = spark.read.parquet(
    "abfss://silver@retaildatasg.dfs.core.windows.net/trans_details/"
)

product_df = spark.read.parquet(
    "abfss://silver@retaildatasg.dfs.core.windows.net/product_details/"
)



Total revenue per product

In [0]:
from pyspark.sql.functions import sum, col

gold_product_sales = silver_df.filter(
    col("payment_status") == "successful"
).groupBy(
    "product_id",
    "product_name"
).agg(
    sum(col("quantity") * col("price")).alias("total_revenue")
)

Revenue per category

In [0]:
gold_category_sales = silver_df.filter(
    col("payment_status") == "successful"
).groupBy("category").agg(
    sum(col("quantity") * col("price")).alias("total_revenue")
)

Daily sales trend

In [0]:
gold_daily_sales = silver_df.filter(
    col("payment_status") == "successful"
).groupBy("transaction_date").agg(
    sum(col("quantity") * col("price")).alias("daily_revenue")
).orderBy("transaction_date")

In [0]:
display(gold_daily_sales)

Payment success vs failure

In [0]:
gold_payment_stats = silver_df.groupBy("payment_status").count()

Top selling products

In [0]:
from pyspark.sql.functions import desc

gold_top_products = silver_df.filter(
    col("payment_status") == "successful"
).groupBy(
    "product_name"
).agg(
    sum(col("quantity")).alias("total_quantity")
).orderBy(desc("total_quantity"))

In [0]:
#revenue once

from pyspark.sql.functions import col, sum

silver_df = silver_df.withColumn(
    "revenue",
    col("quantity") * col("price")
)

Total Revenue

In [0]:
total_revenue = silver_df.filter(
    col("payment_status") == "successful"
).agg(
    sum("revenue").alias("total_revenue")
)

In [0]:
display(total_revenue
        )

Revenue by City

In [0]:
revenue_by_city = silver_df.filter(
    col("payment_status") == "successful"
).groupBy("city").agg(
    sum("revenue").alias("total_revenue")
).orderBy(col("total_revenue").desc())

: Revenue by City + Category

In [0]:
revenue_city_category = silver_df.filter(
    col("payment_status") == "successful"
).groupBy("city", "category").agg(
    sum("revenue").alias("total_revenue")
)

Top Performing Products

In [0]:
from pyspark.sql.functions import sum, desc, col

top_products = silver_df.filter(
    col("payment_status") == "successful"
).groupBy("product_name").agg(
    sum(col("quantity")).alias("total_units_sold"),
    sum(col("quantity") * col("price")).alias("total_revenue")
).orderBy(desc("total_revenue"))

City-wise Transaction Volume

In [0]:
city_volume = silver_df.groupBy("city").agg(
    sum("quantity").alias("total_units"),
    sum(col("quantity") * col("price")).alias("revenue")
)

Average Order Value



In [0]:
aov = silver_df.filter(
    col("payment_status") == "successful"
).agg(
    (sum(col("quantity") * col("price")) / sum("quantity")).alias("avg_order_value")
)

Total Spend per Customer

In [0]:
from pyspark.sql.functions import sum, col

customer_spend = silver_df.filter(
    col("payment_status") == "successful"
).groupBy("customer_id").agg(
    sum(col("quantity") * col("price")).alias("total_spent")
).orderBy(col("total_spent").desc())

Number of Purchases per Customer

In [0]:
customer_orders = silver_df.filter(
    col("payment_status") == "successful"
).groupBy("customer_id").count().withColumnRenamed(
    "count", "total_orders"
)

Repeat vs One-time Customers

In [0]:
from pyspark.sql.functions import count, when

customer_segments = silver_df.filter(
    col("payment_status") == "successful"
).groupBy("customer_id").agg(
    count("*").alias("purchase_count")
)

customer_segments = customer_segments.withColumn(
    "customer_type",
    when(col("purchase_count") > 1, "repeat_customer")
    .otherwise("one_time_customer")
)

Customer Failure Rate

In [0]:
customer_failure = silver_df.groupBy("customer_id").agg(
    sum(when(col("payment_status") == "failed", 1).otherwise(0)).alias("failed_orders"),
    count("*").alias("total_orders")
)

In [0]:
def write_gold(df, table_name):
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .save(f"abfss://gold@retaildatasg.dfs.core.windows.net/{table_name}/")

In [0]:
write_gold(customer_failure, "customer_failure")
write_gold(customer_segments, "customer_segments")
write_gold(customer_orders, "customer_orders")
write_gold(customer_spend, "customer_spend")
write_gold(aov, "average_order_value")
write_gold(city_volume, "city_volume")
write_gold(top_products, "top_products")
write_gold(revenue_city_category, "revenue_city_category")
write_gold(revenue_by_city, "revenue_by_city")
write_gold(total_revenue, "total_revenue")
write_gold(gold_daily_sales, "daily_sales")
write_gold(gold_product_sales, "product_sales")
write_gold(gold_category_sales, "category_sales")
write_gold(gold_payment_stats, "payment_stats")


Dataset with record of all suceessful transactions

In [0]:
gold_transactions = silver_df.filter(
    col("payment_status") == "successful"
)

In [0]:
write_gold(gold_transactions, "gold_transactions")


In [0]:
%sql
CREATE DATABASE IF NOT EXISTS retail_db;

In [0]:
customer_failure.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_db.customer_failure")

In [0]:
%sql
CREATE TABLE retail_db.customer_failure_sql
USING DELTA
LOCATION 'abfss://gold@retaildatasg.dfs.core.windows.net/customer_failure';

In [0]:
customer_segments.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_db.customer_segments")

In [0]:
%sql
CREATE TABLE retail_db.customer_segments_sql
USING DELTA
LOCATION 'abfss://gold@retaildatasg.dfs.core.windows.net/customer_segments';

In [0]:
customer_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_db.customer_orders")

In [0]:
%sql
CREATE TABLE retail_db.customer_orders_sql
USING DELTA
LOCATION 'abfss://gold@retaildatasg.dfs.core.windows.net/customer_orders';

In [0]:
customer_spend.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_db.customer_spend")

In [0]:
%sql
CREATE TABLE retail_db.customer_spend_sql
USING DELTA
LOCATION 'abfss://gold@retaildatasg.dfs.core.windows.net/customer_spend';

In [0]:
aov.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_db.average_order_value")

In [0]:
%sql
CREATE TABLE retail_db.average_order_value_sql
USING DELTA
LOCATION 'abfss://gold@retaildatasg.dfs.core.windows.net/average_order_value';

In [0]:
city_volume.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_db.city_volume")

In [0]:
%sql
CREATE TABLE retail_db.city_volume_sql
USING DELTA
LOCATION 'abfss://gold@retaildatasg.dfs.core.windows.net/city_volume';

In [0]:
top_products.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_db.top_products")

In [0]:
%sql
CREATE TABLE retail_db.top_products_sql
USING DELTA
LOCATION 'abfss://gold@retaildatasg.dfs.core.windows.net/top_products';

In [0]:
revenue_by_city.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_db.revenue_by_city")

In [0]:
%sql
CREATE TABLE retail_db.revenue_by_city_sql
USING DELTA
LOCATION 'abfss://gold@retaildatasg.dfs.core.windows.net/revenue_by_city';

In [0]:
total_revenue.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_db.total_revenue")

In [0]:
%sql
CREATE TABLE retail_db.total_revenue_sql
USING DELTA
LOCATION 'abfss://gold@retaildatasg.dfs.core.windows.net/total_revenue';

In [0]:
gold_category_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_db.category_sales")

In [0]:
%sql
CREATE TABLE retail_db.category_sales_sql
USING DELTA
LOCATION 'abfss://gold@retaildatasg.dfs.core.windows.net/category_sales';

In [0]:
%sql
SELECT * FROM retail_db.category_sales_sql LIMIT 10;

In [0]:
gold_transactions.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_db.trans_table")

In [0]:
%sql
CREATE TABLE retail_db.trans_table_sql
USING DELTA
LOCATION 'abfss://gold@retaildatasg.dfs.core.windows.net/gold_transactions';